# More advanced exercises

Try creating a 3-way, perhaps bringing Gemini into the conversation! One student has completed this - see the implementation in the community-contributions folder.

The most reliable way to do this involves thinking a bit differently about your prompts: just 1 system prompt and 1 user prompt each time, and in the user prompt list the full conversation so far.

Something like:

```python
system_prompt = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie.
"""

user_prompt = f"""
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Alex.
"""
```

Try doing this yourself before you look at the solutions. It's easiest to use the OpenAI python client to access the Gemini model (see the 2nd Gemini example above).

## Additional exercise

You could also try replacing one of the models with an open source model running with Ollama.

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
# Load API keys

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


In [3]:
# Check that all required API keys are available
required_keys = {
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "GROQ_API_KEY": GROQ_API_KEY,
    "OPENROUTER_API_KEY": OPENROUTER_API_KEY,
}

missing_keys = [
    key_name
    for key_name, key_value in required_keys.items()
    if not key_value
]

if missing_keys:
    raise ValueError(
        "Missing API keys: " + ", ".join(missing_keys)
    )


In [4]:
# Create clients

# OpenAI
openai_client = OpenAI(
    api_key=OPENAI_API_KEY,
    timeout=60.0,
    max_retries=1,
)

# Google Gemini through its OpenAI-compatible endpoint
gemini_client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url=(
        "https://generativelanguage.googleapis.com/"
        "v1beta/openai/"
    ),
    timeout=60.0,
    max_retries=1,
)

# Groq through its OpenAI-compatible endpoint
groq_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    timeout=60.0,
    max_retries=1,
)

# OpenRouter through its OpenAI-compatible endpoint
openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    timeout=60.0,
    max_retries=1,
)

# Local Ollama server
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1/",
    api_key="ollama",
    timeout=120.0,
    max_retries=1,
)


In [5]:
# Test every client before running the full conversation

client_tests = [
    {
        "provider": "OpenAI",
        "client": openai_client,
        "model": "gpt-5-mini",
    },
    {
        "provider": "Google Gemini",
        "client": gemini_client,
        "model": "gemini-3.6-flash",
    },
    {
        "provider": "Groq",
        "client": groq_client,
        "model": "openai/gpt-oss-20b",
    },
    {
        "provider": "OpenRouter",
        "client": openrouter_client,
        "model": "openrouter/free",
    },
    {
        "provider": "Local Ollama",
        "client": ollama_client,
        "model": "qwen3:4b",
    },
]


for test in client_tests:
    provider = test["provider"]
    client = test["client"]
    model = test["model"]

    print(f"\nTesting {provider} — {model}")

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": "Reply with only the word OK.",
                }
            ],
        )

        answer = response.choices[0].message.content

        print(f"{provider} is working")
        print(f"Response: {answer}")

    except Exception as error:
        print(f"{provider} failed")
        print(f"Error type: {type(error).__name__}")
        print(f"Error message: {error}")


Testing OpenAI — gpt-5-mini
OpenAI is working
Response: OK

Testing Google Gemini — gemini-3.6-flash
Google Gemini is working
Response: OK

Testing Groq — openai/gpt-oss-20b
Groq is working
Response: OK

Testing OpenRouter — openrouter/free
OpenRouter is working
Response: OK

Testing Local Ollama — qwen3:4b
Local Ollama is working
Response: OK


In [6]:
# Define the five participants

participants = [
    {
        "name": "Alex",
        "provider": "OpenAI",
        "client": openai_client,
        "model": "gpt-5-mini",
        "personality": (
            "You are argumentative and snarky. You disagree with "
            "other participants and challenge their assumptions."
        ),
    },
    {
        "name": "Blake",
        "provider": "Google Gemini",
        "client": gemini_client,
        "model": "gemini-3.6-flash",
        "personality": (
            "You are calm and practical. You try to find a balanced "
            "position and reduce conflict between participants."
        ),
    },
    {
        "name": "Charlie",
        "provider": "Groq",
        "client": groq_client,
        "model": "openai/gpt-oss-20b",
        "personality": (
            "You are highly optimistic about technology. You strongly "
            "defend technological progress and innovation."
        ),
    },
    {
        "name": "Diana",
        "provider": "OpenRouter",
        "client": openrouter_client,
        "model": "openrouter/free",
        "personality": (
            "You are cautious and analytical. You focus on risks, "
            "ethical concerns and unintended consequences."
        ),
    },
    {
        "name": "Eve",
        "provider": "Local Ollama",
        "client": ollama_client,
        "model": "qwen3:4b",
        "personality": (
            "You are curious and independent. You examine both benefits "
            "and risks before reaching your own conclusion."
        ),
    },
]



In [7]:
# Function for calling one participant

def call_participant(participant, conversation):
    """
    Send the complete conversation to one model and receive
    that participant's next response.
    """

    current_name = participant["name"]

    other_names = [
        person["name"]
        for person in participants
        if person["name"] != current_name
    ]

    # One system prompt
    system_prompt = f"""
You are {current_name}.

You are participating in a conversation with:
{", ".join(other_names)}.

Your personality:
{participant["personality"]}

Instructions:
- Remain in character.
- Respond to the existing conversation.
- Do not speak on behalf of the other participants.
- Do not include your name at the beginning.
- Keep your response under 100 words.
"""

    # One user prompt containing the entire conversation
    user_prompt = f"""
You are {current_name}.

The complete conversation so far is:

---------------- CONVERSATION ----------------
{conversation}
----------------------------------------------

Now respond with what you would like to say next as {current_name}.
"""

    request = {
        "model": participant["model"],
        "messages": [
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        "timeout": 60.0,
    }

    if participant["provider"] == "OpenAI":
        request["max_completion_tokens"] = 1000
    else:
        request["max_tokens"] = 1000

    response = participant["client"].chat.completions.create(**request)

    reply = response.choices[0].message.content

    if not reply:
        return "I have nothing to add at this moment."

    return reply.strip()


In [8]:
# Run the conversation

topic = "Is artificial intelligence ultimately good for society?"

conversation = f"""
Discussion topic: {topic}

The five participants are beginning the discussion.
"""

# Three rounds means:
# 5 participants × 3 rounds = 15 model calls
number_of_rounds = 3

print("\nFIVE-MODEL AI CONVERSATION")
print("=" * 70)
print(f"Topic: {topic}")
print("=" * 70)


for round_number in range(1, number_of_rounds + 1):

    print(f"\n{'=' * 25} ROUND {round_number} {'=' * 25}")

    # Everyone receives the same context in round 1, so the opening
    # positions are independent. Later rounds use the live transcript.
    round_start_conversation = conversation

    for participant in participants:

        name = participant["name"]
        provider = participant["provider"]

        print(f"\nCalling {name} using {provider}...")

        try:
            participant_context = (
                round_start_conversation
                if round_number == 1
                else conversation
            )

            reply = call_participant(
                participant=participant,
                conversation=participant_context,
            )

            formatted_message = (
                f"{name} ({provider}): {reply}"
            )

            # Add the response to the shared conversation
            conversation += f"\n\n{formatted_message}"

            print(f"\n{formatted_message}")

        except Exception as error:
            print(
                f"\n{name} ({provider}) could not respond."
            )
            print(f"Error: {error}")



FIVE-MODEL AI CONVERSATION
Topic: Is artificial intelligence ultimately good for society?

========================= ROUND 1 =========================

Calling Alex using OpenAI...

Alex (OpenAI): Calling it simply "good" is lazy. AI is an amplifier: it magnifies whatever institutions, incentives, and values already exist. In weak democracies it deepens surveillance, inequality, and corporate power; with smart regulation and public ownership it can help healthcare, education, and productivity. The real fight isn't about abstract morality — it's about who controls the tech and what rules we enforce. If you think markets will solve this without oversight, enjoy your new digital feudalism.

Calling Blake using Google Gemini...

Blake (Google Gemini): It’s a huge question, and I think the answer really depends on how we manage it. On one hand, AI offers incredible practical benefits—improving healthcare, streamlining tedious tasks, and helping us solve complex problems faster. On the othe

In [9]:
# Display the complete final transcript

print("\n\n")
print("=" * 70)
print("COMPLETE CONVERSATION")
print("=" * 70)
print(conversation)




COMPLETE CONVERSATION

Discussion topic: Is artificial intelligence ultimately good for society?

The five participants are beginning the discussion.


Alex (OpenAI): Calling it simply "good" is lazy. AI is an amplifier: it magnifies whatever institutions, incentives, and values already exist. In weak democracies it deepens surveillance, inequality, and corporate power; with smart regulation and public ownership it can help healthcare, education, and productivity. The real fight isn't about abstract morality — it's about who controls the tech and what rules we enforce. If you think markets will solve this without oversight, enjoy your new digital feudalism.

Blake (Google Gemini): It’s a huge question, and I think the answer really depends on how we manage it. On one hand, AI offers incredible practical benefits—improving healthcare, streamlining tedious tasks, and helping us solve complex problems faster. On the other hand, concerns about job displacement, privacy, and bias are ent

In [10]:
# Use the existing OpenAI participant as the neutral judge.
# This does not add a sixth provider.

def judge_conversation(topic, conversation):
    system_prompt = """
You are a neutral debate judge. Evaluate every participant fairly.
Identify the strongest arguments, important disagreements, and practical
recommendations. Do not favor the OpenAI participant.
"""

    user_prompt = f"""
Debate topic: {topic}

Complete transcript:
{conversation}

Provide these four sections:
1. Strongest arguments
2. Main disagreements
3. Practical recommendations
4. Final balanced conclusion
"""

    response = openai_client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_completion_tokens=1500,
        timeout=60.0,
    )

    return response.choices[0].message.content.strip()


In [11]:
# Generate and display the final synthesis

try:
    final_synthesis = judge_conversation(
        topic=topic,
        conversation=conversation,
    )

    print("\n" + "=" * 70)
    print("FINAL SYNTHESIS")
    print("=" * 70)
    print(final_synthesis)

except Exception as error:
    print("The final synthesis could not be generated.")
    print(f"Error: {error}")



FINAL SYNTHESIS
1. Strongest arguments
- Alex (OpenAI): AI is an amplifier of existing power, institutions, and incentives. Therefore governance choices (who controls models, who profits, what rules exist) determine societal outcome. Concrete policy levers Alex proposes (antitrust, public models/datasets, enforceable audits, data rights, worker protections) are powerful because they target structural drivers rather than surface symptoms.
- Diana (OpenRouter): The incentive analysis—publish‑or‑perish, milestone pressure, headline chasing—explains why risk mitigation often fails in practice. This points to the need for systemic reforms in how research and deployment are rewarded and evaluated.
- Blake (Google Gemini): A pragmatic framing that regulation and innovation are not mutually exclusive; “scaffolding” with enforceable, well‑designed rules can enable responsible progress. This is useful for policy design because it focuses on implementable compromise.
- Charlie (Groq): Emphasizin